In [1]:
import os
import re
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
# Path to our data folder
data_folder = "../data"

# Dictionary to store text from each PDF
documents = {}

# Loop through every PDF in the data folder
for filename in os.listdir(data_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(data_folder, filename)
        reader = PdfReader(filepath)
        
        # Extract all pages
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        
        documents[filename] = text
        print(f"{filename}: {len(text)} characters extracted")

print(f"\nTotal documents loaded: {len(documents)}")

commercial_combined_insurance_policy.pdf: 332617 characters extracted
freight_solutions_policy.pdf: 60192 characters extracted


Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)


legacy_policy.pdf: 215696 characters extracted
office_policy.pdf: 248582 characters extracted
property_owners_policy.pdf: 108299 characters extracted
property_policy.pdf: 101860 characters extracted

Total documents loaded: 6


In [3]:
def clean_text(text):
    # Fix multiple spaces
    text = re.sub(r' +', ' ', text)
    # Normalize multiple newlines to double newline
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Add period after ALL CAPS headings
    text = re.sub(r'([A-Z]{5,})\n', r'\1.\n\n', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""]
)

all_chunks = []

for filename, text in documents.items():
    cleaned = clean_text(text)
    chunks = text_splitter.split_text(cleaned)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

commercial_combined_insurance_policy.pdf: 394 chunks
freight_solutions_policy.pdf: 73 chunks
legacy_policy.pdf: 254 chunks
office_policy.pdf: 303 chunks
property_owners_policy.pdf: 129 chunks
property_policy.pdf: 120 chunks

Total chunks: 1273


In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk['text'] for chunk in all_chunks]

print("Embedding all chunks... please wait")
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

print(f"\nDone! Total embeddings: {embeddings.shape[0]}")
print(f"Embedding dimensions: {embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding all chunks... please wait


Batches:   0%|          | 0/40 [00:00<?, ?it/s]


Done! Total embeddings: 1273
Embedding dimensions: 384


In [6]:
client = chromadb.PersistentClient(path="../data/chromadb")

# Delete old collection if exists and create fresh
try:
    client.delete_collection("insurance_policies")
    print("Old collection deleted")
except:
    print("No existing collection found")

# Create fresh collection
collection = client.get_or_create_collection(
    name="insurance_policies",
    metadata={"hnsw:space": "cosine"}
)

# Prepare data
ids = [f"{chunk['source']}_{chunk['chunk_id']}" for chunk in all_chunks]
texts_list = [chunk['text'] for chunk in all_chunks]
embeddings_list = embeddings.tolist()
metadatas = [{"source": chunk['source'], "chunk_id": chunk['chunk_id']} for chunk in all_chunks]

# Add to ChromaDB
collection.add(
    ids=ids,
    documents=texts_list,
    embeddings=embeddings_list,
    metadatas=metadatas
)

print(f"Total chunks stored: {collection.count()}")

Old collection deleted
Total chunks stored: 1273


In [7]:
def ask_question_test(question, n_results=3):
    # Embed question
    question_embedding = model.encode(question).tolist()
    
    # Retrieve
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )
    
    print(f"Question: {question}\n")
    print("Top 3 retrieved chunks:\n")
    for i in range(n_results):
        print(f"--- Result {i+1} ---")
        print(f"Source: {results['metadatas'][0][i]['source']}")
        print(f"Text: {results['documents'][0][i][:300]}...")
        print()

# Test with questions that failed before
ask_question_test("What is covered under property damage?")


Question: What is covered under property damage?

Top 3 retrieved chunks:

--- Result 1 ---
Source: commercial_combined_insurance_policy.pdf
Text: the Property Insured caused by:
A) pollution or contamination which itself results from any Cover 
insured (other than Cover 10),
B) any Cover insured (other than Cover 10) which itself results 
from pollution or contamination.
6 Property Excluded
Damage to Property which is more specifically insure...

--- Result 2 ---
Source: commercial_combined_insurance_policy.pdf
Text: A) arising from the settlement or movement of made–up ground 
or by coastal or river erosion,
B) occurring as a result of the construction, demolition, structural 13 | Commercial Combined Insurance Policy Wording
alteration or structural repair of any Property at the Premises,
C) arising from normal...

--- Result 3 ---
Source: commercial_combined_insurance_policy.pdf
Text: 1) influence any government or any international 
governmental organisation or
2) put the public or

In [8]:
# Test multiple questions
questions = [
    "What is excluded from storm or flood damage?",
    "Is theft covered under the office policy?",
    "What are the conditions for making a claim?",
    "What is the liability coverage limit?"
]

for q in questions:
    ask_question_test(q)
    print("="*60)

Question: What is excluded from storm or flood damage?

Top 3 retrieved chunks:

--- Result 1 ---
Source: commercial_combined_insurance_policy.pdf
Text: A) arising from nationalisation, confiscation, requisition or 
destruction by order of the government or any public 
authority,
B) arising from cessation of work,
C) i) in the course of theft or attempted theft,
ii) in respect of any Building which is empty or not in use,
directly caused by maliciou...

--- Result 2 ---
Source: legacy_policy.pdf
Text: directly caused by malicious persons not acting on behalf of or 
in connection with any political organisation
4 Storm or flood excluding Damage
 1) attributable solely to change in the water table level (the level 
below which the ground is completely saturated with water)
 2) caused by frost subsi...

--- Result 3 ---
Source: property_policy.pdf
Text: taking part in labour disturbances or malicious persons excluding 
Damage
 1) arising from nationalisation confiscation requisition or 
d

In [9]:
test_questions = [
    "What is excluded from storm or flood damage?",
    "Is theft covered under the office policy?",
    "What are the conditions for making a claim?",
    "What is the liability coverage limit?",
    "What is covered under escape of water?",
    "What are the notification requirements when a loss occurs?",
    "Is accidental damage covered under the property policy?",
    "What documents are required when making a claim?",
    "Are fences and gates covered under storm damage?",
    "What is the excess amount for property damage claims?",
    "Is damage caused by frost covered?",
    "What is covered under malicious damage?",
    "Are empty buildings covered under the policy?",
    "What is the time limit for reporting a claim?",
    "Is damage caused by subsidence covered?"
]

print(f"Total test questions: {len(test_questions)}")

Total test questions: 15


In [10]:
from groq import Groq
from dotenv import load_dotenv
import httpx

load_dotenv()

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY"),
    http_client=httpx.Client(verify=False)
)

def get_rag_answer(question, n_results=3):
    # Embed question
    question_embedding = model.encode(question).tolist()
    
    # Retrieve chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )
    
    # Build context
    context = ""
    retrieved_contexts = []
    for i in range(n_results):
        chunk_text = results['documents'][0][i]
        source = results['metadatas'][0][i]['source']
        context += f"[Source {i+1}: {source}]\n{chunk_text}\n\n"
        retrieved_contexts.append(chunk_text)
    
    # Build prompt
    prompt = f"""You are an expert insurance policy assistant.
Answer the question below using ONLY the context provided.
If the answer is not in the context, say "I could not find this in the provided policy documents."
Always cite which source document your answer comes from.
Be concise, professional, and precise.

Context:
{context}

Question: {question}

Answer:"""

    # Get answer from Groq
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    answer = response.choices[0].message.content
    
    return answer, retrieved_contexts

# Run all 15 questions
print("Running all 15 questions through RAG system...\n")

results_data = []

for i, question in enumerate(test_questions):
    print(f"Processing question {i+1}/15: {question[:50]}...")
    answer, contexts = get_rag_answer(question)
    results_data.append({
        "question": question,
        "answer": answer,
        "contexts": contexts
    })

print("\nDone! All 15 questions answered.")

Running all 15 questions through RAG system...

Processing question 1/15: What is excluded from storm or flood damage?...
Processing question 2/15: Is theft covered under the office policy?...
Processing question 3/15: What are the conditions for making a claim?...
Processing question 4/15: What is the liability coverage limit?...
Processing question 5/15: What is covered under escape of water?...
Processing question 6/15: What are the notification requirements when a loss...
Processing question 7/15: Is accidental damage covered under the property po...
Processing question 8/15: What documents are required when making a claim?...
Processing question 9/15: Are fences and gates covered under storm damage?...
Processing question 10/15: What is the excess amount for property damage clai...
Processing question 11/15: Is damage caused by frost covered?...
Processing question 12/15: What is covered under malicious damage?...
Processing question 13/15: Are empty buildings covered under the po

In [11]:
for i, result in enumerate(results_data):
    print(f"Q{i+1}: {result['question']}")
    print(f"A: {result['answer'][:200]}...")
    print("-"*60)

Q1: What is excluded from storm or flood damage?
A: According to Source 1: commercial_combined_insurance_policy.pdf, the following are excluded from storm or flood damage:
A) attributable solely to change in the water table level,
B) caused by frost, s...
------------------------------------------------------------
Q2: Is theft covered under the office policy?
A: Theft is covered under certain conditions, as stated in [Source 1: office_policy.pdf] and [Source 3: office_policy.pdf]. According to [Source 1: office_policy.pdf], theft achieved by electronic means ...
------------------------------------------------------------
Q3: What are the conditions for making a claim?
A: The conditions for making a claim are outlined in Source 2: office_policy.pdf, under "Claims Conditions" and "1 Making a Claim". Specifically, it states that when an event that could give rise to a cl...
------------------------------------------------------------
Q4: What is the liability coverage limit?
A: The liab

In [18]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset

# Prepare data in RAGAS format
ragas_data = {
    "question": [r["question"] for r in results_data],
    "answer": [r["answer"] for r in results_data],
    "contexts": [r["contexts"] for r in results_data]
}

# Convert to HuggingFace Dataset format (required by RAGAS)
dataset = Dataset.from_dict(ragas_data)

print("Dataset ready for RAGAS evaluation")
print(f"Number of samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

Dataset ready for RAGAS evaluation
Number of samples: 15
Columns: ['question', 'answer', 'contexts']


C:\Users\m209349\AppData\Local\Temp\ipykernel_5972\2487049286.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\m209349\AppData\Local\Temp\ipykernel_5972\2487049286.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [24]:
from groq import Groq
import json

eval_client = Groq(
    api_key=os.getenv("GROQ_API_KEY"),
    http_client=httpx.Client(verify=False)
)

def evaluate_answer(question, answer, contexts):
    context_text = "\n\n".join(contexts)
    
    eval_prompt = f"""You are an evaluation expert. Score the following RAG system answer on two dimensions.

Question: {question}

Retrieved Context:
{context_text}

Generated Answer:
{answer}

Score each dimension from 0.0 to 1.0:

1. FAITHFULNESS: Is every claim in the answer supported by the retrieved context? 
   1.0 = fully supported, 0.5 = partially supported, 0.0 = not supported/hallucinated

2. RELEVANCY: Does the answer actually address the question asked?
   1.0 = directly answers, 0.5 = partially answers, 0.0 = does not answer

Respond ONLY with a JSON object like this:
{{"faithfulness": 0.8, "relevancy": 0.9, "reasoning": "brief explanation"}}"""

    response = eval_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0.0
    )
    
    raw = response.choices[0].message.content
    # Clean and parse JSON
    raw = raw.strip()
    if "```" in raw:
        raw = raw.split("```")[1].replace("json", "").strip()
    
    return json.loads(raw)

# Test on first question
test_eval = evaluate_answer(
    results_data[0]["question"],
    results_data[0]["answer"],
    results_data[0]["contexts"]
)

print(f"Question: {results_data[0]['question']}")
print(f"Faithfulness: {test_eval['faithfulness']}")
print(f"Relevancy: {test_eval['relevancy']}")
print(f"Reasoning: {test_eval['reasoning']}")

Question: What is excluded from storm or flood damage?
Faithfulness: 1.0
Relevancy: 1.0
Reasoning: The answer directly quotes the retrieved context, specifically section 4, which explicitly lists the exclusions from storm or flood damage, making it fully supported and directly relevant to the question asked.


In [25]:
import time

evaluation_results = []

print("Running evaluation across all 15 questions...\n")

for i, result in enumerate(results_data):
    print(f"Evaluating Q{i+1}/15: {result['question'][:50]}...")
    
    try:
        scores = evaluate_answer(
            result["question"],
            result["answer"],
            result["contexts"]
        )
        evaluation_results.append({
            "question": result["question"],
            "answer": result["answer"],
            "faithfulness": scores["faithfulness"],
            "relevancy": scores["relevancy"],
            "reasoning": scores["reasoning"]
        })
        print(f"  Faithfulness: {scores['faithfulness']} | Relevancy: {scores['relevancy']}")
    except Exception as e:
        print(f"  Error: {e}")
        evaluation_results.append({
            "question": result["question"],
            "answer": result["answer"],
            "faithfulness": None,
            "relevancy": None,
            "reasoning": "evaluation failed"
        })
    
    # Small delay to avoid rate limiting
    time.sleep(1)

print("\nEvaluation complete!")

Running evaluation across all 15 questions...

Evaluating Q1/15: What is excluded from storm or flood damage?...
  Faithfulness: 1.0 | Relevancy: 1.0
Evaluating Q2/15: Is theft covered under the office policy?...
  Faithfulness: 0.8 | Relevancy: 0.9
Evaluating Q3/15: What are the conditions for making a claim?...
  Faithfulness: 1.0 | Relevancy: 1.0
Evaluating Q4/15: What is the liability coverage limit?...
  Faithfulness: 1.0 | Relevancy: 1.0
Evaluating Q5/15: What is covered under escape of water?...
  Faithfulness: 0.8 | Relevancy: 0.9
Evaluating Q6/15: What are the notification requirements when a loss...
  Faithfulness: 0.8 | Relevancy: 0.9
Evaluating Q7/15: Is accidental damage covered under the property po...
  Faithfulness: 1.0 | Relevancy: 0.0
Evaluating Q8/15: What documents are required when making a claim?...
  Faithfulness: 1.0 | Relevancy: 1.0
Evaluating Q9/15: Are fences and gates covered under storm damage?...
  Faithfulness: 1.0 | Relevancy: 1.0
Evaluating Q10/15: What

In [26]:
import pandas as pd

# Build results dataframe
df = pd.DataFrame(evaluation_results)

# Summary stats
avg_faithfulness = df['faithfulness'].mean()
avg_relevancy = df['relevancy'].mean()

print("="*60)
print("EVALUATION RESULTS SUMMARY")
print("="*60)
print(f"Average Faithfulness: {avg_faithfulness:.2f}")
print(f"Average Relevancy:    {avg_relevancy:.2f}")
print(f"Overall Score:        {((avg_faithfulness + avg_relevancy)/2):.2f}")
print("="*60)

print("\nPer Question Breakdown:")
print(f"{'Q':<4} {'Faithfulness':<15} {'Relevancy':<12} {'Question'}")
print("-"*60)
for i, row in df.iterrows():
    print(f"Q{i+1:<3} {row['faithfulness']:<15} {row['relevancy']:<12} {row['question'][:45]}...")

print("\nFailed Questions (Relevancy = 0.0):")
failed = df[df['relevancy'] == 0.0]
for i, row in failed.iterrows():
    print(f"  Q{i+1}: {row['question']}")

EVALUATION RESULTS SUMMARY
Average Faithfulness: 0.91
Average Relevancy:    0.71
Overall Score:        0.81

Per Question Breakdown:
Q    Faithfulness    Relevancy    Question
------------------------------------------------------------
Q1   1.0             1.0          What is excluded from storm or flood damage?...
Q2   0.8             0.9          Is theft covered under the office policy?...
Q3   1.0             1.0          What are the conditions for making a claim?...
Q4   1.0             1.0          What is the liability coverage limit?...
Q5   0.8             0.9          What is covered under escape of water?...
Q6   0.8             0.9          What are the notification requirements when a...
Q7   1.0             0.0          Is accidental damage covered under the proper...
Q8   1.0             1.0          What documents are required when making a cla...
Q9   1.0             1.0          Are fences and gates covered under storm dama...
Q10  1.0             0.0          What

In [27]:
# Save to CSV
df.to_csv("../evaluation/ragas_results.csv", index=False)

# Save summary
summary = {
    "avg_faithfulness": avg_faithfulness,
    "avg_relevancy": avg_relevancy,
    "overall_score": (avg_faithfulness + avg_relevancy) / 2,
    "total_questions": len(df),
    "passed": len(df[df['relevancy'] > 0.0]),
    "failed": len(df[df['relevancy'] == 0.0])
}

import json
with open("../evaluation/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results saved to evaluation/ folder")
print(json.dumps(summary, indent=2))

Results saved to evaluation/ folder
{
  "avg_faithfulness": 0.9133333333333334,
  "avg_relevancy": 0.7066666666666667,
  "overall_score": 0.81,
  "total_questions": 15,
  "passed": 11,
  "failed": 4
}
